train it in a virtual env (google colab) to keep your own sanity :p

In [6]:
import pandas as pd
import numpy as np
import tensorflow as tf
from tensorflow import keras
from sklearn.model_selection import train_test_split
import re

In [ ]:
# ===== LOAD DATASET =====
print("Loading dataset...")
df = pd.read_csv('/home/adem/Desktop/projects/chess/model_training/archive (1)/chessData.csv', nrows=5000)  # ++ nrows for a better model later
print(df.head())

Loading dataset...
                                                 FEN Evaluation
0  rnbqkbnr/pppppppp/8/8/4P3/8/PPPP1PPP/RNBQKBNR ...        -10
1  rnbqkbnr/pppp1ppp/4p3/8/4P3/8/PPPP1PPP/RNBQKBN...        +56
2  rnbqkbnr/pppp1ppp/4p3/8/3PP3/8/PPP2PPP/RNBQKBN...         -9
3  rnbqkbnr/ppp2ppp/4p3/3p4/3PP3/8/PPP2PPP/RNBQKB...        +52
4  rnbqkbnr/ppp2ppp/4p3/3p4/3PP3/8/PPPN1PPP/R1BQK...        -26


In [10]:
# ===== PROCESS EVALUATION SCORES =====
def process_eval(eval_str):
    """Convert evaluation string to numeric value"""
    eval_str = str(eval_str).strip()

    # Handle mate scores (e.g., #+6, #-5)
    if '#' in eval_str:
        match = re.search(r'#([+-]?\d+)', eval_str)
        if match:
            moves_to_mate = int(match.group(1))
            # Convert to large positive/negative number
            return 10000 if moves_to_mate > 0 else -10000

    # Handle regular scores (e.g., +6.5, -3.2)
    try:
        return float(eval_str.replace('+', ''))
    except:
        return 0.0

print("\nProcessing evaluation scores...")
df['eval_numeric'] = df['Evaluation'].apply(process_eval)  # Change column name if needed
print(f"Eval score range: {df['eval_numeric'].min()} to {df['eval_numeric'].max()}")



Processing evaluation scores...
Eval score range: -10000.0 to 10000.0


In [11]:
def fen_to_vector(fen):
    """Convert FEN string to 64-dimensional piece vector"""
    # Piece values: empty=0, white pawn=1, black pawn=2, white knight=3, black knight=4, etc.
    piece_map = {
        'P': 1, 'p': 2,
        'N': 3, 'n': 4,
        'B': 5, 'b': 6,
        'R': 7, 'r': 8,
        'Q': 9, 'q': 10,
        'K': 11, 'k': 12
    }

    # Extract board part of FEN (before the first space)
    board_fen = fen.split()[0]
    vector = []

    # Parse FEN: ranks are separated by '/', go through each rank
    for rank in board_fen.split('/'):
        for char in rank:
            if char.isdigit():
                # Empty squares
                for _ in range(int(char)):
                    vector.append(0)
            else:
                # Piece
                vector.append(piece_map.get(char, 0))

    # Normalize to 0-1 range
    return np.array(vector) / 12.0

print("Converting FEN to vectors...")
X = np.array([fen_to_vector(fen) for fen in df['FEN']])  # Change column name if needed
y = df['eval_numeric'].values


Converting FEN to vectors...


In [12]:
print(X[0])
print(y[0])

[0.66666667 0.33333333 0.5        0.83333333 1.         0.5
 0.33333333 0.66666667 0.16666667 0.16666667 0.16666667 0.16666667
 0.16666667 0.16666667 0.16666667 0.16666667 0.         0.
 0.         0.         0.         0.         0.         0.
 0.         0.         0.         0.         0.         0.
 0.         0.         0.         0.         0.         0.
 0.08333333 0.         0.         0.         0.         0.
 0.         0.         0.         0.         0.         0.
 0.08333333 0.08333333 0.08333333 0.08333333 0.         0.08333333
 0.08333333 0.08333333 0.58333333 0.25       0.41666667 0.75
 0.91666667 0.41666667 0.25       0.58333333]
-10.0


In [13]:
y = np.clip(y / 5.0, -1, 1)

print(f"Input shape: {X.shape}")
print(f"Output shape: {y.shape}")


Input shape: (5000, 64)
Output shape: (5000,)


In [14]:
# ===== TRAIN TEST SPLIT =====
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(f"\nTrain set: {X_train.shape}")
print(f"Test set: {X_test.shape}")


# ===== BUILD MODEL =====
print("\nBuilding model...")
model = keras.Sequential([
    keras.layers.Dense(256, activation='relu', input_shape=(64,)),
    keras.layers.Dropout(0.2),
    keras.layers.Dense(128, activation='relu'),
    keras.layers.Dropout(0.2),
    keras.layers.Dense(64, activation='relu'),
    keras.layers.Dense(1, activation='linear')  # Output: evaluation score
])

model.compile(
    optimizer='adam',
    loss='mse',
    metrics=['mae']
)

print(model.summary())



Train set: (4000, 64)
Test set: (1000, 64)

Building model...


/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:93: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 256)            │        16,640 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 128)            │        32,896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 1)              │            65 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 57,857 (226.00 KB)

 Trainable params: 57,857 (226.00 KB)

 Non-trainable params: 0 (0.00 B)

None


In [15]:
# ===== TRAIN MODEL =====
print("\nTraining model...")
history = model.fit(
    X_train, y_train,
    epochs=10,
    batch_size=32,
    validation_split=0.1,
    verbose=1
)



Training model...
Epoch 1/10
113/113 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step - loss: 0.8457 - mae: 0.8625 - val_loss: 0.7328 - val_mae: 0.7807
Epoch 2/10
113/113 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 0.6976 - mae: 0.7488 - val_loss: 0.6033 - val_mae: 0.6517
Epoch 3/10
113/113 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - loss: 0.5491 - mae: 0.6247 - val_loss: 0.5565 - val_mae: 0.6108
Epoch 4/10
113/113 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 0.4841 - mae: 0.5743 - val_loss: 0.5075 - val_mae: 0.5791
Epoch 5/10
113/113 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 0.4141 - mae: 0.5106 - val_loss: 0.4999 - val_mae: 0.5343
Epoch 6/10
113/113 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 0.3967 - mae: 0.4899 - val_loss: 0.4755 - val_mae: 0.5371
Epoch 7/10
113/113 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - loss: 0.3645 - mae: 0.4652 - val_loss: 0.4671 - val_mae: 0.5312
Epoch 8/10
113/113 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - loss: 0.3461 - mae: 0.4480 - val_loss: 0.4518 - val_mae: 0.5082
Epoch 9/10
113/113 ━━━━━━━━━━━━━━━━━━

In [16]:
# ===== EVALUATE =====
print("\nEvaluating on test set...")
test_loss, test_mae = model.evaluate(X_test, y_test)
print(f"Test Loss (MSE): {test_loss:.4f}")
print(f"Test MAE: {test_mae:.4f}")




Evaluating on test set...
32/32 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 0.4503 - mae: 0.5046
Test Loss (MSE): 0.4265
Test MAE: 0.4888


In [17]:
# ===== SAVE MODEL =====
print("\nSaving model...")
model.save('chess_eval_model.h5')
print("Model saved as 'chess_eval_model.h5'")


Saving model...
Model saved as 'chess_eval_model.h5'
